# Seasonal Agriculture Performance Analysis
**VOIS AICTE Batch 1 2026–2027 – Major Project**

This notebook analyzes the supplied agricultural dataset to identify seasonal patterns, trends, relationships, differences, and data-driven recommendations, following the project brief.

## 1. Problem Statement
Agricultural performance varies with seasonal environmental conditions, farming practices, resource availability and market conditions. The analysis investigates how performance changes across Kharif, Rabi and Zaid seasons and identifies meaningful patterns and variations.

## 2. Objectives
- Explore and understand the dataset
- Clean and prepare the data
- Compare agricultural performance across seasons
- Investigate environmental/resource relationships with outcomes
- Compare crops, irrigation methods and regions
- Identify unusual patterns and significant differences
- Produce evidence-based conclusions and recommendations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal, chi2_contingency

df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 3. Data Quality and Cleaning
The dataset contains 4,000 records and 28 columns. Missing values occur in Rainfall, Soil Moisture and Yield. There are no duplicate rows. Numeric values were checked for negative/impossible values. Missing rainfall and soil-moisture values were imputed using season-level medians; missing yield values were imputed using crop-season medians to preserve seasonal/crop context.

In [ ]:
clean = df.copy()

for col in ["Rainfall_mm", "Soil_Moisture_pct"]:
    clean[col] = clean.groupby("Season")[col].transform(
        lambda s: s.fillna(s.median())
    )

clean["Yield_Tonnes_Ha"] = clean.groupby(
    ["Crop", "Season"]
)["Yield_Tonnes_Ha"].transform(
    lambda s: s.fillna(s.median())
)

clean["Profit_Margin_pct"] = np.where(
    clean["Revenue_INR"] != 0,
    clean["Profit_INR"] / clean["Revenue_INR"] * 100,
    np.nan
)

print("Remaining missing values:", int(clean.isna().sum().sum()))

## 4. Seasonal Performance

In [ ]:
season_order = ["Kharif", "Rabi", "Zaid"]

season_summary = clean.groupby("Season").agg(
    Farms=("Farm_ID","count"),
    Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha","mean"),
    Avg_Production_Tonnes=("Production_Tonnes","mean"),
    Avg_Revenue_INR=("Revenue_INR","mean"),
    Avg_Cost_INR=("Total_Cost_INR","mean"),
    Avg_Profit_INR=("Profit_INR","mean"),
    Median_Profit_INR=("Profit_INR","median"),
    Avg_Water_Used_m3=("Water_Used_m3","mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Disease_Risk_pct=("Disease_Pest_Risk_pct","mean")
).reindex(season_order)

season_summary["Profitable_Farm_%"] = (
    clean.assign(Profitable=clean["Profit_INR"] > 0)
    .groupby("Season")["Profitable"].mean().mul(100)
    .reindex(season_order)
)

season_summary.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(season_order, season_summary["Avg_Profit_INR"]/1000)
ax.axhline(0, linewidth=1)
ax.set_title("Average Profit by Season")
ax.set_ylabel("Average profit (₹ thousand)")
plt.show()

### Key seasonal findings
- **Kharif** has the highest average yield (5.64 t/ha), average profit (₹178,915) and profitable-farm rate (57.79%).
- **Rabi** is intermediate on yield and profitability.
- **Zaid** has the lowest average yield (4.67 t/ha), negative average profit (−₹24,805) and the lowest profitable-farm rate (35.52%).
- Disease/pest risk is highest in Kharif (54.47%), while Zaid has the lowest average risk (38.22%).
- The differences across seasons are statistically significant for yield, profit, water efficiency and disease risk using Kruskal–Wallis tests (all p < 0.001).

## 5. Crop × Season Analysis

In [ ]:
crop_season_yield = clean.pivot_table(
    index="Crop", columns="Season",
    values="Yield_Tonnes_Ha", aggfunc="mean"
).reindex(columns=season_order)

crop_season_profit = clean.pivot_table(
    index="Crop", columns="Season",
    values="Profit_INR", aggfunc="mean"
).reindex(columns=season_order)

display(crop_season_yield.round(2))
display(crop_season_profit.round(0))

### Crop findings
- Sugarcane has by far the highest yield and average profit across seasons in this dataset.
- Chilli is the second strongest economic performer and remains highly profitable across all three seasons.
- Cotton and Groundnut are more favorable in Kharif than Zaid.
- Wheat, Rice and Maize show negative average profit across all three seasons, with the weakest results generally occurring in Zaid.
- Therefore, crop choice should be considered together with season rather than evaluating crops in isolation.

## 6. Irrigation and Resource Use

In [ ]:
irrigation = clean.groupby(
    ["Season","Irrigation_Method"]
).agg(
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean")
).round(2)

irrigation

### Irrigation findings
- Drip records the highest average yield among irrigation methods in Kharif and Rabi.
- Sprinkler records the highest average yield in Zaid.
- Rainfed has high water-efficiency values because water use is comparatively low; this should not automatically be interpreted as the highest production or profitability.
- Irrigation choice should therefore balance yield, water availability and economic return rather than optimizing a single metric.

## 7. Regional Analysis

In [ ]:
state_profit = clean.pivot_table(
    index="State", columns="Season",
    values="Profit_INR", aggfunc="mean"
).reindex(columns=season_order)

state_profit.round(0)

### Regional findings
- Kharif is the strongest profit season for every state in the dataset.
- Punjab and Karnataka show positive average Zaid profit, unlike most other states.
- Maharashtra has comparatively strong Rabi profitability.
- The state-level differences suggest that seasonal planning should account for regional conditions instead of applying a single strategy everywhere.

## 8. Relationship Analysis

In [ ]:
corr_cols = [
    "Rainfall_mm","Avg_Temperature_C","Humidity_pct",
    "Soil_Moisture_pct","Nitrogen_kg_ha","Fertilizer_kg_ha",
    "Seed_Quality_Score","Yield_Tonnes_Ha","Profit_INR",
    "Water_Used_m3","Disease_Pest_Risk_pct"
]
corr = clean[corr_cols].corr(method="spearman")
corr.round(2)

### Relationship findings
- Yield has a positive association with water-efficiency and a smaller positive association with rainfall and soil moisture.
- Profit is positively associated with yield and water-efficiency and has a weaker positive association with rainfall.
- Fertilizer quantity has a weak negative association with profit in this dataset; this is an association, not proof that fertilizer reduces profit.
- Correlation does not establish causation, so operational decisions should be validated with domain knowledge and additional controlled data.

## 9. Statistical Testing

In [ ]:
groups_yield = [clean.loc[clean["Season"]==s,"Yield_Tonnes_Ha"] for s in season_order]
groups_profit = [clean.loc[clean["Season"]==s,"Profit_INR"] for s in season_order]
groups_eff = [clean.loc[clean["Season"]==s,"Water_Efficiency_t_per_1000m3"] for s in season_order]
groups_risk = [clean.loc[clean["Season"]==s,"Disease_Pest_Risk_pct"] for s in season_order]

tests = {
    "Yield": kruskal(*groups_yield),
    "Profit": kruskal(*groups_profit),
    "Water efficiency": kruskal(*groups_eff),
    "Disease/pest risk": kruskal(*groups_risk),
}
tests

In [ ]:
contingency = pd.crosstab(
    clean["Season"], clean["Profit_INR"] > 0
)
chi2_stat, chi2_p, dof, expected = chi2_contingency(contingency)

print("Chi-square test: profitable vs non-profitable by season")
print("chi2 =", round(chi2_stat,2), "p =", chi2_p)
contingency

### Statistical interpretation
The Kruskal–Wallis results indicate statistically significant differences between seasons for yield, profit, water efficiency and disease/pest risk (p < 0.001). The chi-square test also indicates that profitability status is associated with season (p < 0.001). Statistical significance indicates that the observed group differences are unlikely to be explained by random variation alone under the test assumptions; it does not by itself establish causation.

## 10. Conclusions and Recommendations
1. **Prioritize Kharif planning** where feasible because it has the strongest overall yield and profitability in this dataset.
2. **Evaluate crop-season combinations**, with Sugarcane and Chilli showing the strongest economic performance.
3. **Use irrigation strategically**: Drip performs strongly for yield in Kharif/Rabi, while Zaid results favor Sprinkler for yield in this dataset.
4. **Monitor Zaid economics carefully** because its average profit is negative and its profitable-farm rate is lowest.
5. **Integrate climate and pest-risk monitoring**, especially for Kharif where disease/pest risk is highest.
6. **Avoid one-size-fits-all regional planning** because state-level seasonal profitability differs.
7. Treat correlations as signals for further investigation rather than causal conclusions.

## 11. Future Scope
- Build predictive models for yield and profit.
- Add multi-year historical data to distinguish seasonal effects from year effects.
- Include crop-specific market price and input-cost forecasts.
- Develop an interactive dashboard for farmers and planners.
- Test scenario simulations for rainfall, irrigation and input changes.
- Add satellite/soil/weather data for more granular regional analysis.